In [ ]:
# Requisitos: numpy, matplotlib, rainflow (pip install numpy matplotlib rainflow)
import math
import numpy as np
import matplotlib.pyplot as plt
import rainflow   # python-rainflow: ASTM E1049-85 (3 puntos)

def astm_e1049_propio(series):
    """
    Implementación propia (solo para verificación cruzada) del algoritmo Rainflow siguiendo la norma ASTM E1049-85.
    Recorre el historial transitorio de esfuerzos identificando inversiones (picos/valles)
    y extrae los rangos correspondientes del bucle.

    Devuelve: Una lista de tuplas con la estructura (rango, esfuerzo_medio, conteo_ciclos)
    """
    pts = list(series)
    stack = []
    cycles = []

    for p in pts:
        stack.append(p)
        while len(stack) >= 3:
            X = abs(stack[-1] - stack[-2])
            Y = abs(stack[-2] - stack[-3])

            if X >= Y:
                # Si cumple la condición de cierre de ciclo de la norma ASTM
                if len(stack) == 3:
                    Y_rango = abs(stack[0] - stack[1])
                    Y_media = (stack[0] + stack[1]) / 2.0
                    cycles.append((Y_rango, Y_media, 0.5))
                    stack.pop(0)
                else:
                    Y_rango = abs(stack[-2] - stack[-3])
                    Y_media = (stack[-2] + stack[-3]) / 2.0
                    cycles.append((Y_rango, Y_media, 1.0))
                    stack.pop(-2)
                    stack.pop(-2)
            else:
                break

    # Procesamiento del residuo terminal (ciclos que quedan abiertos al acabarse la señal)
    while len(stack) >= 2:
        rango = abs(stack[0] - stack[1])
        media = (stack[0] + stack[1]) / 2.0
        cycles.append((rango, media, 0.5))
        stack.pop(0)

    return cycles


def ciclos_rainflow(series):
    """Ciclos (rango, media, conteo) con la librería rainflow (ASTM E1049-85)."""
    return [(rng, mean, count) for rng, mean, count, i0, i1 in rainflow.extract_cycles(series)]

def _norm(c):
    return sorted((round(r, 6), round(m, 6), n) for r, m, n in c)

# ==============================================================================
# 1. PARAMETRIZACIÓN PROPIEDADES DEL MATERIAL Y GEOMETRÍA
# ==============================================================================
S_ut = 460.0       # Resistencia última a la tracción del material (MPa)
S_e_mod = 142.18   # Límite de resistencia a la fatiga completamente modificado (MPa)
d = 30.0           # Diámetro de la sección transversal del eje (mm)
Kf = 1.539         # Factor de concentración de esfuerzos por fatiga a Flexión
Kfs = 1.369        # Factor de concentración de esfuerzos por fatiga a Torsión

# Historial de carga transitoria provisto (Unidades en N*m)
momentos = [0, 340, -260, 420, -30, 330, -400, 200, -200, 0]
torques  = [0, 323, -247, 400, -28, 314, -381, 190, -190, 0]

# ==============================================================================
# 2. TRANSFORMACIÓN A ESFUERZOS EQUIVALENTES DE VON MISES SIGNADO
# ==============================================================================
c = d / 2.0
I = (math.pi * (d**4)) / 64.0
J = (math.pi * (d**4)) / 32.0

von_mises_original = []
for M, T in zip(momentos, torques):
    # Esfuerzo normal por flexión (sigma = M * c / I) incluyendo el concentrador Kf
    sig = (Kf * (M * 1000.0) * c) / I
    # Esfuerzo cortante por torsión (tau = T * c / J) incluyendo el concentrador Kfs
    tau = (Kfs * (T * 1000.0) * c) / J
    # Magnitud escalar del esfuerzo de Von Mises triaxial equivalente
    vm_mag = math.sqrt(sig**2 + 3.0 * (tau**2))
    # Se le asigna el signo del esfuerzo normal para discriminar tracción (+) y compresión (-)
    signo = np.sign(sig) if sig != 0 else 1.0
    von_mises_original.append(signo * vm_mag)

# ==============================================================================
# 3. ROTACIÓN / REORDENAMIENTO PARA EL MÉTODO SIMPLIFICADO (QUICK RAINFLOW)
# ==============================================================================
# Filtramos los ceros de los extremos para hallar la ubicación del pico máximo absoluto efectivo
valores_efectivos = von_mises_original[1:-1]
idx_pico_max = int(np.argmax(np.abs(valores_efectivos))) + 1

# Reordenamiento cíclico: el historial se corta y se une empezando y terminando en el máximo absoluto
von_mises_reordenado = von_mises_original[idx_pico_max:-1] + von_mises_original[1:idx_pico_max + 1]
if von_mises_reordenado[0] != von_mises_reordenado[-1]:
    von_mises_reordenado.append(von_mises_reordenado[0])

# ==============================================================================
# 4. DETERMINACIÓN DE PARÁMETROS DE LA CURVA DE FATIGA DE BASQUIN (S-N)
# ==============================================================================
s_1000 = 0.9 * S_ut
b_exp = -(1.0 / 3.0) * math.log10(s_1000 / S_e_mod)
a_basq = s_1000 / (1000.0**b_exp)

# ==============================================================================
# 5. ALGORITMO DE PROCESAMIENTO DE ESPECTROS Y DAÑO DE PALMGREN-MINER
# ==============================================================================
def procesar_espectro_completo(lista_esfuerzos, es_simplificado=False):
    # Extraer los ciclos iniciales con el rainflow (ASTM E1049-85)
    ciclos_raw = ciclos_rainflow(lista_esfuerzos)
    # Verificación cruzada: la librería y la implementación propia deben coincidir
    assert _norm(ciclos_raw) == _norm(astm_e1049_propio(lista_esfuerzos)), 'rainflow y algoritmo propio difieren'

    # Método simplificado (historia cerrada en el máximo absoluto)
    if es_simplificado:
        # La historia reordenada empieza y termina en el máximo absoluto: el ciclo mayor aparece
        # como dos medios ciclos idénticos (0.5 + 0.5) que se agrupan en un ciclo completo
        agrupados = {}
        for rango, media, n in ciclos_raw:
            clave = (round(rango, 6), round(media, 6))
            agrupados[clave] = agrupados.get(clave, 0.0) + n
        assert all(abs(n - round(n)) < 1e-9 for n in agrupados.values()), 'quedaron medios ciclos sin pareja'
        ciclos_procesados = [(r, m, float(round(n))) for (r, m), n in agrupados.items()]
    else:
        ciclos_procesados = ciclos_raw

    sa_arr, sm_arr, count_arr, dano_arr = [], [], [], []
    tabla_datos = []
    dano_acumulado_bloque = 0.0

    for rango, media, n in ciclos_procesados:
        sa = rango / 2.0
        # Corrección de esfuerzo medio usando la ecuación lineal de Goodman (solo si Sm > 0)
        factor_goodman = (1.0 - (media / S_ut)) if media > 0 else 1.0
        sar = sa / factor_goodman

        # Evaluación del daño acumulado frente a la curva S-N
        if sar > S_e_mod:
            N_fallo = (sar / a_basq) ** (1.0 / b_exp)
            daño_i = n / N_fallo
        else:
            N_fallo = float('inf')
            daño_i = 0.0

        dano_acumulado_bloque += daño_i

        sa_arr.append(sa)
        sm_arr.append(media)
        count_arr.append(n)
        dano_arr.append(daño_i)
        tabla_datos.append([sa, media, sar, n, N_fallo, daño_i])

    return np.array(sm_arr), np.array(sa_arr), np.array(count_arr), np.array(dano_arr), tabla_datos, dano_acumulado_bloque

# Ejecución independiente de los dos enfoques analíticos
sm_m1, sa_m1, count_m1, dano_m1, tabla_m1, dano_total_m1 = procesar_espectro_completo(von_mises_original, es_simplificado=False)
sm_m2, sa_m2, count_m2, dano_m2, tabla_m2, dano_total_m2 = procesar_espectro_completo(von_mises_reordenado, es_simplificado=True)

# Ley de Miner inversa: cálculo de la vida útil medida en repeticiones del bloque completo
vida_bloques_m1 = 1.0 / dano_total_m1 if dano_total_m1 > 0 else float('inf')
vida_bloques_m2 = 1.0 / dano_total_m2 if dano_total_m2 > 0 else float('inf')

# ==============================================================================
# 6. GENERACIÓN Y GUARDADO DE GRÁFICOS COMPARATIVOS 3D
# ==============================================================================
dx, dy = 12.0, 12.0  # Ancho volumétrico de las barras en la matriz (MPa)

# FIGURA I: MATRIZ TRIDIMENSIONAL DE CONTEO DE CICLOS
fig_c, (ax1, ax2) = plt.subplots(1, 2, subplot_kw={'projection': '3d'}, figsize=(16, 7))

# Gráfico Subplot 1: Método Estricto ASTM
ax1.bar3d(sm_m1, sa_m1, np.zeros_like(sm_m1), dx, dy, count_m1, color=plt.cm.viridis(count_m1), edgecolor='black', alpha=0.8)
ax1.set_title('Método 1: ASTM E1049 Estricto\n(Matriz de Conteo con Medios Ciclos Residuales)', fontsize=11)
ax1.set_xlabel(r'$\sigma_m$ (MPa)')
ax1.set_ylabel(r'$S_a$ (MPa)')
ax1.set_zlabel('Conteo ($n$)')
ax1.set_zlim(0, 1.1)
ax1.view_init(elev=20, azim=-55)

# Gráfico Subplot 2: Método Simplificado Ajustado
ax2.bar3d(sm_m2, sa_m2, np.zeros_like(sm_m2), dx, dy, count_m2, color=plt.cm.viridis(count_m2), edgecolor='black', alpha=0.8)
ax2.set_title('Método 2: Conteo Simplificado (Quick Rainflow)\n(Matriz de Conteo - Todo Cerrado Coherente a 1.0)', fontsize=11)
ax2.set_xlabel(r'$\sigma_m$ (MPa)')
ax2.set_ylabel(r'$S_a$ (MPa)')
ax2.set_zlabel('Conteo ($n$)')
ax2.set_zlim(0, 1.1)
ax2.view_init(elev=20, azim=-55)

plt.tight_layout()
plt.savefig('comparativa_3d_ciclos_corregido.png', dpi=150)
plt.close(fig_c)

# FIGURA II: MATRIZ TRIDIMENSIONAL DE DISTRIBUCIÓN DE DAÑO RELATIVO DE MINER
fig_d, (ax3, ax4) = plt.subplots(1, 2, subplot_kw={'projection': '3d'}, figsize=(16, 7))
max_dano_escala = max(np.max(dano_m1), np.max(dano_m2)) if max(np.max(dano_m1), np.max(dano_m2)) > 0 else 1.0

# Gráfico Subplot 3: Distribución de Daño Método 1
ax3.bar3d(sm_m1, sa_m1, np.zeros_like(sm_m1), dx, dy, dano_m1, color=plt.cm.plasma(dano_m1 / max_dano_escala), edgecolor='black', alpha=0.8)
ax3.set_title('Método 1: ASTM E1049 Estricto\n(Distribución de Daño Acumulado de Miner)', fontsize=11)
ax3.set_xlabel(r'$\sigma_m$ (MPa)')
ax3.set_ylabel(r'$S_a$ (MPa)')
ax3.set_zlabel('Daño Relativo ($n/N$)')
ax3.view_init(elev=20, azim=-55)

# Gráfico Subplot 4: Distribución de Daño Método 2
ax4.bar3d(sm_m2, sa_m2, np.zeros_like(sm_m2), dx, dy, dano_m2, color=plt.cm.plasma(dano_m2 / max_dano_escala), edgecolor='black', alpha=0.8)
ax4.set_title('Método 2: Conteo Simplificado (Quick Rainflow)\n(Distribución de Daño Acumulado de Miner)', fontsize=11)
ax4.set_xlabel(r'$\sigma_m$ (MPa)')
ax4.set_ylabel(r'$S_a$ (MPa)')
ax4.set_zlabel('Daño Relativo ($n/N$)')
ax4.view_init(elev=20, azim=-55)

plt.tight_layout()
plt.savefig('comparativa_3d_dano_corregido.png', dpi=150)
plt.close(fig_d)

# ==============================================================================
# 7. VISUALIZACIÓN FORMATEADA DE REPORTES NUMÉRICOS
# ==============================================================================
def imprimir_reporte_consola(titulo, tabla, dano_total, vida_bloques):
    print("=" * 95)
    print(f" {titulo.upper()} ")
    print("=" * 95)
    print(f"{'Sa (MPa)':<12} | {'Sm (MPa)':<12} | {'Sar-Good (MPa)':<16} | {'Conteo (n)':<12} | {'N Fallo':<15} | {'Daño (n/N)':<12}")
    print("-" * 95)
    for fila in tabla:
        n_fallo_str = f"{fila[4]:.2e}" if fila[4] != float('inf') else "Infinito (inf)"
        dano_str = f"{fila[5]:.4e}" if fila[5] > 0 else "0.0000e+00"
        print(f"{fila[0]:<12.2f} | {fila[1]:<12.2f} | {fila[2]:<16.2f} | {fila[3]:<12.1f} | {n_fallo_str:<15} | {dano_str:<12}")
    print("-" * 95)
    print(f"▶ Daño total acumulado por bloque: {dano_total:.6e}")
    print(f"▶ VIDA ÚTIL ESTIMADA: {vida_bloques:,.2f} bloques de carga" if vida_bloques != float('inf') else "▶ VIDA ÚTIL ESTIMADA: Infinita")
    print("=" * 95 + "\n")

# Imprimir las tablas de resultados consolidadas directamente en la terminal
print("=== PROCESAMIENTO Y CORRECCIÓN ANALÍTICA DE CICLOS COMPLETADA ===")
imprimir_reporte_consola("Método 1: ASTM E1049 Estricto (Historial Original)", tabla_m1, dano_total_m1, vida_bloques_m1)
imprimir_reporte_consola("Método 2: Conteo Simplificado (Quick Rainflow - Todo Cerrado a 1.0)", tabla_m2, dano_total_m2, vida_bloques_m2)